In [11]:
%pip install sqlalchemy "psycopg[binary]" python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [28]:
from sqlalchemy import text
from database import engine
import pandas as pd

print("Libraries imported successfully!")

Libraries imported successfully!


In [29]:
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT version()"))
        print("PostgreSQL connection successful!")
        print("Database version:", result.scalar())

except Exception as e:
    print("Database connection failed!")
    print("Error:", e)

PostgreSQL connection successful!
Database version: PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [5]:
from sqlalchemy import inspect
from database import engine

inspector = inspect(engine)
tables = inspector.get_table_names(schema="public")

print(f"Found {len(tables)} table(s) in schema 'public':")
for t in tables:
    print(f" - {t}")

Found 1 table(s) in schema 'public':
 - sanskrit


In [22]:
from sqlalchemy import text
from database import engine

with engine.begin() as conn:  # begin() auto-commits on success
    conn.execute(text("ALTER TABLE tts_metadata RENAME TO sanskrit"))
    # optional: tidy up the identity sequence and primary key names
    #conn.execute(text("ALTER SEQUENCE old_table_name_ui_id_seq RENAME TO new_table_name_ui_id_seq"))
    #conn.execute(text("ALTER INDEX old_table_name_pkey RENAME TO new_table_name_pkey"))

print("Table renamed")

Table renamed


In [10]:
from sqlalchemy import inspect
from database import engine
import pandas as pd

table_name = "sanskrit"  # Change if using another table

inspector = inspect(engine)
columns = inspector.get_columns(table_name)

df_cols = pd.DataFrame([{
    "Column Name": col["name"],
    "Type": str(col["type"]),
    "Nullable": col["nullable"],
    "Default": col.get("default")
} for col in columns])

print(f"Columns in '{table_name}':")
df_cols

Columns in 'sanskrit':


,Column Name,Type,Nullable,Default
0,ui_id,INTEGER,False,None
1,record_date,DATE,True,None
2,language,VARCHAR(100),True,None
3,gender,VARCHAR(30),True,None
4,source,VARCHAR(255),True,None
5,source_links,TEXT,True,None
6,license_tag,VARCHAR(100),True,None
7,data_description,TEXT,True,None
8,vedic_accent,VARCHAR(100),True,None
9,file_name,TEXT,True,None


In [ ]:
from sqlalchemy import text
from database import engine

with engine.begin() as conn:   # auto-commits on success
 #   conn.execute(text("DROP TABLE IF EXISTS sanskrit"))
    conn.execute(text("ALTER TABLE sanskrit DROP COLUMN remark"))

Table deleted


In [11]:
with engine.connect() as connection:
    query = text("SELECT COUNT(*) FROM sanskrit")
    total_records = connection.execute(query).scalar()

    print("Total records stored:", total_records)

Total records stored: 456


In [12]:
import pandas as pd
from database import engine

table_name = "sanskrit"

# Query the first 5 records
query = f"SELECT * FROM {table_name} LIMIT 5"
df = pd.read_sql(query, con=engine)

# Display in notebook
df

,ui_id,record_date,language,gender,source,source_links,license_tag,data_description,vedic_accent,file_name,...,meter,style,multispeaker,categories,processed_through,researched_by,status,speaker_id,text_links,validated_by
0,1,2026-12-08,Sanskrit,Male,You Tube,https://youtu.be/6KgoqAuvnUo?si=95TylSpSBlvJ__wj,None,Vedic Chanting | S. Rajagopalsharma | Sanskrit...,None,1. Aditya_Bhakti_chanting_m_1\n\n2. Ganapathi ...,...,Mix Meters,Vedic Chant,No,None,None,Raghav,Discard,Aditya_Bhakthi_01,https://www.shdvef.com/wp-content/uploads/2019...,None
1,2,2026-09-26,Sanskrit,Male,You Tube,https://www.youtube.com/watch?v=Bh9Es2iHSOw&li...,Standard YouTube License,आनोभद्राः । ஆநோபத்ரா: । Aanobhadra: । Rigveda ...,vedic_text,A_NO_Bhadra_Suktam_M,...,Anustubh,Vedic Chant,No,None,Sloka_wise_split,Sanket,Not Used,Swami02,https://sanskritdocuments.org/doc_veda/Anobhad...,None
2,3,2026-08-29,Sanskrit,Male,You Tube,https://www.youtube.com/watch?v=i1DBiu4rR74&li...,Standard YouTube License,Aaranyaka 1 - Aruna Prasna - Suryanamaskara Ma...,None,Aaranyaka_1,...,Adhyavasitam,Vedic Chant,Yes,None,Sloka_wise_split,Sanket,Discard,Arunam,https://sanskritdocuments.org/doc_veda/taittir...,Sanket
3,4,2026-08-29,Sanskrit,Male,You Tube,https://www.youtube.com/watch?v=aZC1pMdVjTI&li...,Standard YouTube License,Aaranyaka 2 - Swadhyaya Bramhanam\nchannel : V...,None,Aaranyaka_2,...,None,Vedic Chant,Yes,None,Sloka_wise_split,Sanket,Discard,Arunam,https://sanskritdocuments.org/doc_veda/taittir...,Sanket
4,5,2026-12-08,Sanskrit,Male,Website,https://www.prapatti.com/slokas/mp3/abhiitista...,Standard YouTube License,Abhiitistava\nVedaanta Desikan\n (Desika St...,None,Abhiitistava_m,...,Mix Meters,Stotra,No,None,Line_wise_Split,Nikhil,Done,Vedanta_Desikan_01,https://www.prapatti.com/slokas/sanskrit/desik...,Nikhil


In [26]:
from sqlalchemy import text
from database import engine

with engine.connect() as connection:
    total_records = connection.execute(
        text("SELECT COUNT(*) FROM sanskrit")
    ).scalar_one()

print(f"Total records remaining: {total_records}")

if total_records == 0:
    print("✅ Ready to upload the CSV again.")
else:
    print("⚠️ The table still contains records.")

Total records remaining: 456
⚠️ The table still contains records.
